# Bab 6. Berkas, Pengodean, dan Penanganan Galat

Kode pendamping buku *Python untuk Machine Learning dan Data
Science*. Jalankan selnya berurutan dari atas, sebab sebagian
sel memakai peubah dari sel sebelumnya.

Notebook ini dibangkitkan dari naskah buku. Jangan disunting di
sini, sunting listing pada berkas `.tex` lalu bangkitkan ulang.

## 1. Menulis lalu membaca

In [ ]:
with open("uji.txt", "w", encoding="utf-8") as f:
    f.write("baris satu\n")
    f.write("baris dua\n")

with open("uji.txt", encoding="utf-8") as f:
    isi = f.read()

print(repr(isi))
print(f.closed)

Keluaran yang diharapkan:

```
'baris satu\nbaris dua\n'
True
```

## 2. Menyusuri baris demi baris

In [ ]:
with open("besar.txt", encoding="utf-8") as f:
    for nomor, baris in enumerate(f, start=1):
        if "galat" in baris:
            print(nomor, baris.rstrip())

## 3. Satu teks, dua tafsiran

In [ ]:
teks = "Rp5 juta \u2014 naik 10%"     # \u2014 tanda pisah
print(len(teks))

data = teks.encode("utf-8")
print(len(data))
print(data.decode("ascii", "backslashreplace"))

Keluaran yang diharapkan:

```
19
21
Rp5 juta \xe2\x80\x94 naik 10%
```

## 4. Dibaca dengan pengodean yang keliru

In [ ]:
salah = data.decode("latin-1")
print(len(salah))

benar = data.decode("utf-8")
print(len(benar), benar == teks)

Keluaran yang diharapkan:

```
21
19 True
```

## 5. Sebagian pengodean lain menolak

In [ ]:
with open("utf.txt", encoding="ascii") as f:
    f.read()

Keluaran yang diharapkan:

```
UnicodeDecodeError: 'ascii' codec can't decode byte
0xe2 in position 4
```

## 6. Cara cepat yang salah

In [ ]:
baris = 'P1,"Nastar, spesial",85000'
print(baris.split(","))

Keluaran yang diharapkan:

```
['P1', '"Nastar', ' spesial"', '85000']
```

## 7. Cara yang benar

In [ ]:
import csv

with open("data.csv", encoding="utf-8", newline="") as f:
    for baris in csv.reader(f):
        print(baris)

Keluaran yang diharapkan:

```
['id', 'nama', 'harga']
['P1', 'Nastar, spesial', '85000']
```

## 8. Membaca sebagai dict

In [ ]:
with open("data.csv", encoding="utf-8", newline="") as f:
    for r in csv.DictReader(f):
        print(r["nama"], int(r["harga"]))

Keluaran yang diharapkan:

```
Nastar 85000
Kastengel 95000
```

## 9. Menyimpan dan memuat JSON

In [ ]:
import json
from pathlib import Path

catatan = {"produk": "Nastar", "jumlah": 3,
           "harga": 85000, "kanal": ["toko", "online"]}

Path("catatan.json").write_text(
    json.dumps(catatan, indent=2, ensure_ascii=False),
    encoding="utf-8")

kembali = json.loads(
    Path("catatan.json").read_text(encoding="utf-8"))

print(kembali == catatan)

Keluaran yang diharapkan:

```
True
```

## 10. Operasi jalur

In [ ]:
from pathlib import Path

p = Path("data") / "mentah" / "penjualan.csv"

print(p)
print(p.parent)
print(p.name, p.stem, p.suffix)
print(p.with_suffix(".json"))

Path("data/mentah").mkdir(parents=True, exist_ok=True)
print(p.exists())
print([str(x) for x in Path("data").rglob("*.csv")])

Keluaran yang diharapkan:

```
data/mentah/penjualan.csv
data/mentah
penjualan.csv penjualan .csv
data/mentah/penjualan.json
True
['data/mentah/penjualan.csv']
```

## 11. Empat bagian penanganan galat

In [ ]:
def uji(bagi):
    try:
        h = 10 / bagi
    except ZeroDivisionError:
        print("except dijalankan")
        return "dari except"
    else:
        print("else dijalankan")
        return "dari else"
    finally:
        print("finally selalu dijalankan")

print(uji(2))
print(uji(0))

Keluaran yang diharapkan:

```
else dijalankan
finally selalu dijalankan
dari else
except dijalankan
finally selalu dijalankan
dari except
```

## 12. Galat yang tersamar

In [ ]:
def hitung(data):
    try:
        return sum(data) / len(dataa)   # salah ketik
    except:
        return 0

print(hitung([1, 2, 3]))

Keluaran yang diharapkan:

```
0
```

## 13. Menolak dengan jelas

In [ ]:
class GalatData(ValueError):
    """Data tidak memenuhi syarat."""

def rerata(data):
    if not data:
        raise GalatData("data kosong")
    if any(x < 0 for x in data):
        raise GalatData("ada nilai negatif")
    return sum(data) / len(data)

## 14. Pengukur waktu

In [ ]:
from contextlib import contextmanager
import time

@contextmanager
def ukur(nama):
    mulai = time.perf_counter()
    try:
        yield
    finally:
        lama = (time.perf_counter() - mulai) * 1000
        print(f"{nama}: {lama:.1f} ms")

with ukur("perulangan"):
    s = sum(i*i for i in range(200_000))

Keluaran yang diharapkan:

```
perulangan: 15.2 ms
```